1. Carga de datos:

In [91]:
import numpy as np

# Ejecutar el archivo fuente para registrar las funciones
%run ../src/retail-sales-analysis.py

# 1. Cargar datos
ruta = '../data/retail_sales_dataset.csv'
datos = cargar_datos(ruta)
datos

array([(   1, '2023-11-24', 'CUST001', 'Male', 34, 'Beauty', 3,  50,  150),
       (   2, '2023-02-27', 'CUST002', 'Female', 26, 'Clothing', 2, 500, 1000),
       (   3, '2023-01-13', 'CUST003', 'Male', 50, 'Electronics', 1,  30,   30),
       (   4, '2023-05-21', 'CUST004', 'Male', 37, 'Clothing', 1, 500,  500),
       (   5, '2023-05-06', 'CUST005', 'Male', 30, 'Beauty', 2,  50,  100),
       (   6, '2023-04-25', 'CUST006', 'Female', 45, 'Beauty', 1,  30,   30),
       (   7, '2023-03-13', 'CUST007', 'Male', 46, 'Clothing', 2,  25,   50),
       (   8, '2023-02-22', 'CUST008', 'Male', 30, 'Electronics', 4,  25,  100),
       (   9, '2023-12-13', 'CUST009', 'Male', 63, 'Electronics', 2, 300,  600),
       (  10, '2023-10-07', 'CUST010', 'Female', 52, 'Clothing', 4,  50,  200),
       (  11, '2023-02-14', 'CUST011', 'Male', 23, 'Clothing', 2,  50,  100),
       (  12, '2023-10-30', 'CUST012', 'Male', 35, 'Beauty', 3,  25,   75),
       (  13, '2023-08-05', 'CUST013', 'Male', 22, 'Elect

2. Exploración inicial:

In [92]:
# Inspeccionar dimensiones y tipos de datos
print(f"Dimensiones del dataset: {datos.shape}")
print(f"Estructura detectada (dtype): {datos.dtype}")

Dimensiones del dataset: (1000,)
Estructura detectada (dtype): [('f0', '<i8'), ('f1', '<U10'), ('f2', '<U8'), ('f3', '<U6'), ('f4', '<i8'), ('f5', '<U11'), ('f6', '<i8'), ('f7', '<i8'), ('f8', '<i8')]


3. Se lleva a cabo una traducción de los códigos resultantes de hacer 'dtype', para saber con exactitud el tipo de datos presentes en el dataset

In [93]:
# Mapeo de nombres reales de las columnas (según el dataset de Retail de Kaggle)
categorias = [
    "Transaction ID", "Date", "Customer ID", "Gender", "Age", 
    "Product Category", "Quantity", "Price per Unit", "Total Amount"
]

print("--- DICCIONARIO DE DATOS TÉCNICO-FUNCIONAL ---")
# Encabezados de la tabla
encabezado = f"{'ID':<4} | {'Nombre Real':<18} | {'Código':<8} | {'Tipo':<12} | {'Ejemplo Real'}"
print(encabezado)
print("-" * len(encabezado))

# Registro de muestra (fila 0)
primera_fila = datos[0]

# Iterar sobre la estructura del dataset
for i, id_tec in enumerate(datos.dtype.names):

    # Obtener el código (ej: <i8)
    codigo = datos.dtype[id_tec].str
    
    # Traducir el código a un tipo legible
    if 'i' in codigo:
        tipo = "Entero"
    elif 'U' in codigo:
        tipo = "Texto"
    elif 'f' in codigo:
        tipo = "Decimal"
    else:
        tipo = "Otro"
    
    # Obtener y limpiar el dato de ejemplo
    valor = primera_fila[id_tec]
    if isinstance(valor, (bytes, np.bytes_)):
        valor = valor.decode('utf-8')
    
    # El nombre real se toma de nuestra lista manual
    columna = categorias[i] if i < len(categorias) else "Desconocido"
    
    print(f"{id_tec:<4} | {columna:<18} | {codigo:<8} | {tipo:<12} | {valor}")

print("-" * len(encabezado))

--- DICCIONARIO DE DATOS TÉCNICO-FUNCIONAL ---
ID   | Nombre Real        | Código   | Tipo         | Ejemplo Real
------------------------------------------------------------------
f0   | Transaction ID     | <i8      | Entero       | 1
f1   | Date               | <U10     | Texto        | 2023-11-24
f2   | Customer ID        | <U8      | Texto        | CUST001
f3   | Gender             | <U6      | Texto        | Male
f4   | Age                | <i8      | Entero       | 34
f5   | Product Category   | <U11     | Texto        | Beauty
f6   | Quantity           | <i8      | Entero       | 3
f7   | Price per Unit     | <i8      | Entero       | 50
f8   | Total Amount       | <i8      | Entero       | 150
------------------------------------------------------------------


4. Chequear extremos del dataset

In [94]:
# Visualizar primeros y últimos lugares en el dataset
print("\nPrimeros 3 registros:")
print(datos[:3])
print("\nÚltimos 3 registros:")
print(datos[-3:])


Primeros 3 registros:
[(1, '2023-11-24', 'CUST001', 'Male', 34, 'Beauty', 3,  50,  150)
 (2, '2023-02-27', 'CUST002', 'Female', 26, 'Clothing', 2, 500, 1000)
 (3, '2023-01-13', 'CUST003', 'Male', 50, 'Electronics', 1,  30,   30)]

Últimos 3 registros:
[( 998, '2023-10-29', 'CUST998', 'Female', 23, 'Beauty', 4, 25, 100)
 ( 999, '2023-12-05', 'CUST999', 'Female', 36, 'Electronics', 3, 50, 150)
 (1000, '2023-04-12', 'CUST1000', 'Male', 47, 'Electronics', 4, 30, 120)]


5. Chequear si existen datos faltantes

In [95]:
# Analizar valores faltantes
print("--- VALORES FALTANTES ---")

# Recorrer la lista de nombres de columnas
for i, nombre in enumerate(datos.dtype.names):
    
    # Creamos un contador para datos faltantes
    vacia = 0
    
    # Recorrer cada fila del dataset para revisar esa columna específica
    for fila in datos:
        dato = fila[i]
        
        # Verificar si el dato es un texto vacío o un byte vacío(Numpy podría haber cargado de ambas maneras)
        if dato == '' or dato == b'': 
            vacia = vacia + 1
            
    # Imprimir el resultado de la columna actual antes de pasar a la siguiente
    print(f"Columna {nombre} (índice {i}): {vacia} valores faltantes")

--- VALORES FALTANTES ---
Columna f0 (índice 0): 0 valores faltantes
Columna f1 (índice 1): 0 valores faltantes
Columna f2 (índice 2): 0 valores faltantes
Columna f3 (índice 3): 0 valores faltantes
Columna f4 (índice 4): 0 valores faltantes
Columna f5 (índice 5): 0 valores faltantes
Columna f6 (índice 6): 0 valores faltantes
Columna f7 (índice 7): 0 valores faltantes
Columna f8 (índice 8): 0 valores faltantes


7. Chequear "Falsos Nulos" (NaN, None, "null")

In [96]:
print("--- VALORES FALSOS NULOS ---")

for i, nombre in enumerate(datos.dtype.names):
    falsos_nulos = 0
    
    for fila in datos:
        dato = fila[i]
        
        # Convertimos el dato a texto y a minúsculas para atrapar 'None', 'none', 'NULL', etc.
        dato_texto = str(dato).lower()
        
        if dato_texto == 'none' or dato_texto == 'nan' or dato_texto == 'null':
            falsos_nulos = falsos_nulos + 1
            
    print(f"Columna {nombre} (índice {i}): {falsos_nulos} nulos encontrados")

--- VALORES FALSOS NULOS ---
Columna f0 (índice 0): 0 nulos encontrados
Columna f1 (índice 1): 0 nulos encontrados
Columna f2 (índice 2): 0 nulos encontrados
Columna f3 (índice 3): 0 nulos encontrados
Columna f4 (índice 4): 0 nulos encontrados
Columna f5 (índice 5): 0 nulos encontrados
Columna f6 (índice 6): 0 nulos encontrados
Columna f7 (índice 7): 0 nulos encontrados
Columna f8 (índice 8): 0 nulos encontrados


6. Chequear outliers

In [97]:
# Extraer los montos totales de la columna 8
montos_ventas = []
for fila in datos:
    # fila[8] es el "Total Amount" (Monto Total de la Venta)
    valor_venta = float(fila[8]) 
    montos_ventas.append(valor_venta)

# Se crea un arreglo para trabajar con Numpy
ventas_array = np.array(montos_ventas)

# Encontrar los puntos de corte (Cuartiles)
q1 = np.percentile(ventas_array, 25) # El 25% de las ventas más bajas
q3 = np.percentile(ventas_array, 75) # El 75% de las ventas (donde empiezan las ventas altas)

# Calcular el "Ancho de la Venta Normal" (IQR)
rango_ventas = q3 - q1

# Establecer los límites de lo que aceptamos como "Venta Común"
limite_superior = q3 + (1.5 * rango_ventas)
limite_inferior = q1 - (1.5 * rango_ventas)

# Buscar las ventas que se salen de los límites
ventas_fuera_de_rango = []
for i in ventas_array:
    # Si la venta es mayor al límite superior o menor al inferior
    if i > limite_superior or i < limite_inferior:
        ventas_fuera_de_rango.append(i)

# Mostrar resultados
print("--- ANÁLISIS DE VENTAS TOTALES (OUTLIERS) ---")
print(f"Límite máximo para una venta normal: ${limite_superior:.2f}")
print(f"Límite mínimo para una venta normal: ${limite_inferior:.2f}")
print(f"Cantidad de ventas detectadas como atípicas: {len(ventas_fuera_de_rango)}")

if len(ventas_fuera_de_rango) > 0:
    print(f"Algunas de estas ventas fueron: {ventas_fuera_de_rango[:5]}")

--- ANÁLISIS DE VENTAS TOTALES (OUTLIERS) ---
Límite máximo para una venta normal: $2160.00
Límite mínimo para una venta normal: $-1200.00
Cantidad de ventas detectadas como atípicas: 0


### Nota: Interpretación del Límite Inferior Negativo

Al realizar el análisis de valores atípicos (outliers) mediante el método del Rango Intercuartiles (IQR), se observa que el **Límite Inferior** arroja un valor negativo. A continuación, se explica por qué esto es correcto y qué significa para nuestro análisis:

* **¿Por qué es negativo?**: El cálculo es puramente matemático ($Q1 - 1.5 \times IQR$). Si la dispersión de los precios es muy grande comparada con los valores bajos, la fórmula resta más de lo que hay, cruzando la barrera del cero.
* **Sentido de Realidad**: En el contexto de este dataset de Retail, los montos totales de venta no pueden ser menores a cero.
* **Conclusión**: Un límite inferior negativo no es un error. Simplemente nos confirma que **no existen ventas atípicas por ser "demasiado bajas"**. Todas las ventas pequeñas del dataset entran dentro del rango considerado como "normal".

7 Limpieza de datos

In [98]:
# Aunque no haya nulos hoy, esta función es para maximizar la seguridad
datos = limpiar_datos(datos) 
print("Pipeline: Datos verificados y listos para el análisis.")

Pipeline: Datos verificados y listos para el análisis.


7. Exporar si existen duplicados

In [99]:
print("--- BUSCANDO REGISTROS REPETIDOS ---")

filas_vistas = []
duplicados_encontrados = []

for fila in datos:
    # Convertir la fila a una lista
    fila_lista = list(fila)
    
    if fila_lista in filas_vistas:
        duplicados_encontrados.append(fila_lista)
    else:
        filas_vistas.append(fila_lista)

print(f"Cantidad de filas duplicadas: {len(duplicados_encontrados)}")

if len(duplicados_encontrados) > 0:
    print("Se encontraron datos repetidos. Es necesario eliminarlos para no alterar las estadísticas.")

--- BUSCANDO REGISTROS REPETIDOS ---
Cantidad de filas duplicadas: 0


7. Exploración de datos

In [100]:
categorias_analisis = ['Beauty', 'Clothing', 'Electronics']

# Listas para guardar los resultados
nombres_cats = []
resultados_totales = []

print("--- ANÁLISIS DE VENTAS DIARIAS POR CATEGORÍA ---")

for cat in categorias_analisis:
    # Filtrar los datos por categoría
    datos_cat = filtrar_por_categoria(datos, cat)
    
    if len(datos_cat) > 0:
        # Crear un diccionario para agrupar ventas por fecha
        ventas_por_dia = {}
        
        for fila in datos_cat:
            fecha = fila[1] # Columna 1 es la fecha
            monto = float(fila[8]) # Columna 8 es el monto
            
            # Si el día ya existe, suma monto al total del día
            if fecha in ventas_por_dia:
                ventas_por_dia[fecha] = ventas_por_dia[fecha] + monto
            # Si no existe, se agrega
            else:
                ventas_por_dia[fecha] = monto
        
        # Declarar lista con los totales de cada día
        totales_diarios = list(ventas_por_dia.values())
        
        # Calcular promedio diario
        promedio_diario = np.mean(totales_diarios)
        total_categoria = np.sum(totales_diarios)
        
        # Guardar nombre y total para comparar al final
        nombres_cats.append(cat)
        resultados_totales.append(total_categoria)
        
        print(f"\nCategoría: {cat}")
        print(f"  - Total acumulado: ${total_categoria:,.2f}")
        print(f"  - Promedio de ventas diarias: ${promedio_diario:,.2f}")
        print(f"  - Cantidad de días con ventas: {len(totales_diarios)}")

# --- IDENTIFICACIÓN DE MAYORES Y MENORES ---
print("\n--- IDENTIFICACIÓN FINAL ---")

# valores extremos basados en el total acumulado
mayor_v = max(resultados_totales)
menor_v = min(resultados_totales)

# Categorías correspondientes almax y min de ventas totales
cat_mayor = nombres_cats[resultados_totales.index(mayor_v)]
cat_menor = nombres_cats[resultados_totales.index(menor_v)]

# Resultado final
print(f"Tras el análisis, la categoría con mayores ventas es '{cat_mayor}' y la categoría con menores ventas es '{cat_menor}'.")

--- ANÁLISIS DE VENTAS DIARIAS POR CATEGORÍA ---

Categoría: Beauty
  - Total acumulado: $143,515.00
  - Promedio de ventas diarias: $703.50
  - Cantidad de días con ventas: 204

Categoría: Clothing
  - Total acumulado: $155,580.00
  - Promedio de ventas diarias: $670.60
  - Cantidad de días con ventas: 232

Categoría: Electronics
  - Total acumulado: $156,905.00
  - Promedio de ventas diarias: $716.46
  - Cantidad de días con ventas: 219

--- IDENTIFICACIÓN FINAL ---
Tras el análisis, la categoría con mayores ventas es 'Electronics' y la categoría con menores ventas es 'Beauty'.


8. Manipulación de Datos

In [101]:
# Mostrar cuántos registros quedan post limpieza
print(f"Registros listos para procesar: {len(datos)}")

# OPERACIONES MATEMÁTICAS ADICIONALES
# Estadísticas base
total_ventas, promedio, maximo, minimo = obtener_estadisticas(datos)

# Resta: Diferencia entre la venta más alta y la más baja
diferencia_max_min = maximo - minimo

# Multiplicación: Proyección de ventas al doble
proyeccion_doble = total_ventas * 2

# División: Calcular el aporte promedio de cada registro al total
aporte_por_registro = total_ventas / len(datos)

print("\n--- ESTADÍSTICAS ADICIONALES ---")
print(f"Suma Total de Ventas: ${total_ventas:,.2f}")          # Suma
print(f"Diferencia (Max - Min): ${diferencia_max_min:,.2f}")   # Resta
print(f"Proyección Doble: ${proyeccion_doble:,.2f}")           # Multiplicación
print(f"Aporte promedio por fila: ${aporte_por_registro:,.2f}") # División

# FILTRADO Y MUESTRA POR CATEGORÍA
# Filtra los datos para mostrar solo las ventas de una categoría específica
categoria_especifica = 'Beauty'
datos_filtrados_final = filtrar_por_categoria(datos, categoria_especifica)

if len(datos_filtrados_final) > 0:
    print(f"\n--- MOSTRANDO VENTAS PARA: {categoria_especifica} ---")
    
    # Recorrer lista para mostrar los datos en pantalla
    for venta in datos_filtrados_final:
        print(venta)
        
    print(f"\nSe encontraron {len(datos_filtrados_final)} registros de esta categoría.")
else:
    print(f"\nNo se encontraron registros para {categoria_especifica}")

# Dimensiones finales
print(f"\nDimensiones finales del arreglo (Filas, Columnas): {datos.shape}")

Registros listos para procesar: 1000

--- ESTADÍSTICAS ADICIONALES ---
Suma Total de Ventas: $456,000.00
Diferencia (Max - Min): $1,975.00
Proyección Doble: $912,000.00
Aporte promedio por fila: $456.00

--- MOSTRANDO VENTAS PARA: Beauty ---
(1, '2023-11-24', 'CUST001', 'Male', 34, 'Beauty', 3, 50, 150)
(5, '2023-05-06', 'CUST005', 'Male', 30, 'Beauty', 2, 50, 100)
(6, '2023-04-25', 'CUST006', 'Female', 45, 'Beauty', 1, 30, 30)
(12, '2023-10-30', 'CUST012', 'Male', 35, 'Beauty', 3, 25, 75)
(21, '2023-01-14', 'CUST021', 'Female', 50, 'Beauty', 1, 500, 500)
(25, '2023-12-26', 'CUST025', 'Female', 64, 'Beauty', 1, 50, 50)
(27, '2023-08-03', 'CUST027', 'Female', 38, 'Beauty', 2, 25, 50)
(28, '2023-04-23', 'CUST028', 'Female', 43, 'Beauty', 1, 500, 500)
(30, '2023-10-29', 'CUST030', 'Female', 39, 'Beauty', 3, 300, 900)
(32, '2023-01-04', 'CUST032', 'Male', 30, 'Beauty', 3, 30, 90)
(35, '2023-08-05', 'CUST035', 'Female', 58, 'Beauty', 3, 300, 900)
(36, '2023-06-24', 'CUST036', 'Male', 52, 'B